# 061 — Clasificación y representación visual

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Representación visual:** una imagen es un tensor `H×W×3` semánticamente inútil en crudo;
clasificar exige transformarla en un **embedding** donde la distancia refleje semántica.
Antes de 2012 las features se diseñaban a mano (Sobel, HOG, SIFT); desde AlexNet se
aprenden con **convoluciones**: filtros pequeños compartidos en toda la imagen que, al
apilarse, componen jerarquías (bordes → texturas → partes → objetos).

**Clasificación:** sobre el embedding `z`, una capa lineal produce logits `s = W·z + b`
y softmax los convierte en distribución: `softmax(s)_k = exp(s_k)/Σ exp(s_j)`. La pérdida
es la entropía cruzada. Softmax siempre suma 1: confianza alta **no** implica que la
entrada pertenezca a alguna clase conocida.

**Augmentación:** transformar las imágenes de entrenamiento (recorte, espejo, color)
codifica invariancias — solo si respetan la semántica (un `6` espejado no es un `6`).
**Transfer learning:** un tronco preentrenado en ImageNet + cabeza lineal nueva permite
clasificar con cientos de ejemplos en lugar de millones.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Zona uniforme: cada columna aporta `50·(-1)+50·(+1)` (y análogos con ±2),
todo se cancela → salida **0**: no hay borde. Parche `90|10|10`: columna izquierda clara y
derecha oscura → `(-90-180-90) + (10+20+10) = -320`. El **signo negativo** indica la
polaridad del borde (claro→oscuro), la magnitud indica su fuerza.

**Ejercicio 2.** `exp = [2.718, 20.086, 2.718]`, suma `25.522` →
`softmax ≈ [0.106, 0.787, 0.106]`: gana `perro`. Sumar una constante c a todos los logits
multiplica numerador y denominador por `exp(c)`, que se cancela: softmax es **invariante a
traslaciones** de los logits, la clase ganadora no cambia.

**Ejercicio 3.** (a) Válida: un gato espejado sigue siendo un gato. (b) Inválida: `6`
espejado deja de ser un `6` y `2`/`5` se confunden. (c) Inválida: espejar una flecha de
"giro a la derecha" produce la clase contraria — enseñaría exactamente el error que se
quiere evitar.

**Ejercicio 4.** Las claves (`kind`, `seed`, `evidence`, `limitations`, …) son el contrato
estable; los valores derivados de la semilla son muestreo. La comparación en código está
debajo.


In [ ]:
result = run_lab("perception", seed=61)
assert result["kind"] == "perception"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — convolución verificada con código
filtro = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
parche_uniforme = [[50, 50, 50], [50, 50, 50], [50, 50, 50]]
parche_borde = [[90, 10, 10], [90, 10, 10], [90, 10, 10]]

def conv(parche, filtro):
    return sum(parche[i][j] * filtro[i][j] for i in range(3) for j in range(3))

print("uniforme:", conv(parche_uniforme, filtro))   # 0 → sin borde
print("borde:   ", conv(parche_borde, filtro))      # -320 → borde claro→oscuro


In [ ]:
# Ejercicio 2 — softmax e invariancia a traslación
import math

def softmax(logits):
    exps = [math.exp(x) for x in logits]
    s = sum(exps)
    return [e / s for e in exps]

print([round(p, 3) for p in softmax([1.0, 3.0, 1.0])])
print([round(p, 3) for p in softmax([11.0, 13.0, 11.0])])  # idéntico: invariante


In [ ]:
# Ejercicio 4 — contrato vs muestreo
r61 = run_lab("perception", seed=61)
r62 = run_lab("perception", seed=62)
print("mismas claves:", sorted(r61.keys()) == sorted(r62.keys()))
print("mismo contenido:", r61 == r62)


## Reflexión

1. El laboratorio `perception` es determinista con semilla fija. ¿Qué parte de un pipeline
   real de clasificación visual (augmentación, inicialización, orden de batches) rompería
   ese determinismo y cómo lo registrarías?
2. Si tu clasificador de 3 clases devuelve softmax `[0.98, 0.01, 0.01]` ante una imagen que
   no pertenece a ninguna clase, ¿qué mecanismo añadirías antes de automatizar decisiones?
3. ¿Por qué un embedding preentrenado en ImageNet puede rendir peor en imágenes médicas o
   satelitales, y qué evidencia pedirías antes de confiar en él?
